In [1]:
import os

def load_text_from_folder(folder_path: str) -> str:
    """
    指定フォルダ内の .txt ファイルをすべて読み込み、
    1つの文字列として結合して返す。

    :param folder_path: 読み込み対象フォルダのパス
    :return: 結合されたテキスト
    """
    texts = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".txt"):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, "r", encoding="utf-8") as f:
                texts.append(f.read())

    return "\n".join(texts)


# 使い方（あなたの環境に合わせてパスを変更）
#folder_path = r"C:\Users\USER\Documents\RAG_system\semantic_chunk\txt"
#full_document = load_text_from_folder(folder_path)

#full_document


In [2]:
import torch
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

device = "cuda" if torch.cuda.is_available() else "cpu"

model_path = r"C:\Users\USER\Documents\RAG_system\semantic_chunk\models--BAAI--bge-m3"
model_kwargs = {"device": device}
encode_kwargs = {"normalize_embeddings": True}  # Cosine Similarity

embedding_model = HuggingFaceBgeEmbeddings(
    model_name=model_path,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)



C:\Users\USER\AppData\Local\Temp\ipykernel_125724\23068688.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceBgeEmbeddings
C:\Users\USER\AppData\Local\Temp\ipykernel_125724\23068688.py:10: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceBgeEmbeddings(
C:\Users\USER\miniforge3\envs\semantic_chunk\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. S

In [3]:
import numpy as np
import re
from typing import List, Dict
from langchain_core.documents import Document


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# ---------------------------------------------------------
# 0. 日本語見出し検出（階層対応）
# ---------------------------------------------------------
def detect_japanese_heading(line: str) -> bool:
    if len(line) == 0:
        return False

    # 既存の番号付き見出し
    patterns = [
        r"^\d+\)", r"^（\d+）", r"^[①②③④⑤⑥⑦⑧⑨⑩]", r"^\d+\."
    ]
    if any(re.match(p, line) for p in patterns):
        return True

    # ★ 追加：日付行（ブログ形式）
    if re.match(r"^\d{4}年\d{1,2}月\d{1,2}日", line):
        return True

    if re.match(r"^on\s*\d{1,2}月\s*\d{1,2},\s*\d{4}", line):
        return True

    # ★ 追加：タイトル行（長めの名詞句）
    if len(line) <= 60 and not re.search(r"[。．！？]", line):
        # 名詞句っぽい行は見出し扱い
        return True

    return False



# ---------------------------------------------------------
# 1. 見出し単位で分割（階層対応）
# ---------------------------------------------------------
def parse_japanese_sections(text: str) -> List[Dict]:
    lines = text.split("\n")

    blocks = []
    buffer = []
    current_section = "＜＜あたま＞＞"

    def normalize(s: str) -> str:
        # 空白・全角半角を除去して比較を安定化
        return re.sub(r"\s+", "", s)

    def flush():
        nonlocal buffer, current_section
        if buffer:
            blocks.append({
                "section": current_section,
                "content": "\n".join(buffer).strip()
            })
            buffer = []

    for raw in lines:
        line = raw.strip()
        if not line:
            continue

        # 見出し判定
        if detect_japanese_heading(line):

            # ★ 同じ見出しなら絶対に分割しない（今回の追加）
            if normalize(line) == normalize(current_section):
                # 見出しを本文として扱う（必要なら削除してもOK）
                buffer.append(line)
                continue

            # ★ 違う見出しなら新しいセクションとして扱う
            flush()
            current_section = line
            continue

        buffer.append(line)

    flush()
    return blocks


# ---------------------------------------------------------
# 2. 改造版 HybridSemanticChunker
# ---------------------------------------------------------
class HybridSemanticChunker:
    def __init__(
        self,
        embedding_model,
        breakpoint_ratio=0.20,
        merge_threshold=0.75,
        max_tokens=500
    ):
        self.embedding_model = embedding_model
        self.breakpoint_ratio = breakpoint_ratio
        self.merge_threshold = merge_threshold
        self.max_tokens = max_tokens

    def _split_paragraphs(self, text: str) -> List[str]:
    # 空行で段落分割しない
        return [text]

        

    def _embed(self, items: List[str]) -> np.ndarray:
        return np.array(self.embedding_model.embed_documents(items))

    def _compute_similarities(self, embeddings: np.ndarray) -> List[float]:
        return [
            cosine_similarity(embeddings[i], embeddings[i - 1])
            for i in range(1, len(embeddings))
        ]

    def _find_breakpoints(self, sims: List[float]) -> List[int]:
        if len(sims) == 0:
            return []

        threshold = np.quantile(sims, self.breakpoint_ratio)
        return [i for i, s in enumerate(sims) if s < threshold]

    def _too_large(self, text: str) -> bool:
        return len(text) > self.max_tokens

    def split_text(self, text: str) -> List[str]:
        paragraphs = self._split_paragraphs(text)
        final_chunks = []

        for para in paragraphs:
            items = re.split(r"(?<=[。．！？])\s*", para)
            items = [s.strip() for s in items if s.strip()]

            if len(items) == 1:
                final_chunks.append(items[0])
                continue

            embeddings = self._embed(items)
            sims = self._compute_similarities(embeddings)
            breakpoints = self._find_breakpoints(sims)

            chunks = []
            current = items[0]

            for i in range(1, len(items)):
                sim = sims[i - 1]

                # 類似度が高い → 同じ意味段落
                if sim > self.merge_threshold:
                    current += "\n" + items[i]
                else:
                    # ブレークポイント
                    if (i - 1) in breakpoints:
                        chunks.append(current)
                        current = items[i]
                    else:
                        current += "\n" + items[i]

                # チャンクが大きすぎる場合は強制分割
                if self._too_large(current):
                    chunks.append(current)
                    current = ""

            if current:
                chunks.append(current)

            final_chunks.extend(chunks)

        return final_chunks

    # ---------------------------------------------------------
    # 3. 見出し単位でチャンク化
    # ---------------------------------------------------------
    def split_by_toc(self, text: str) -> List[Dict]:
        toc_blocks = parse_japanese_sections(text)
        results = []
    
        for block in toc_blocks:
            # ★ センテンス分割・類似度分割を使わず、見出しごとに1チャンクにする
            chunk_with_heading = f"{block['section']}\n\n{block['content']}"
    
            results.append({
                "section": block["section"],
                "chunk_index": 0,
                "content": chunk_with_heading
            })
    
        return results



    # ---------------------------------------------------------
    # 4. Document 化
    # ---------------------------------------------------------
    def create_documents(self, text) -> List[Document]:
        # text が list の場合は結合
        if isinstance(text, list):
            text = "\n".join(text)
    
        toc_chunks = self.split_by_toc(text)
        docs = []
    
        for c in toc_chunks:
            docs.append(
                Document(
                    page_content=c["content"],
                    metadata={
                        "section": c["section"],
                        "chunk_index": c["chunk_index"],
                    }
                )
            )
    
        return docs



In [4]:
from pprint import pprint

folder_path = r"C:\Users\USER\Documents\RAG_system\semantic_chunk\txt"
full_document = load_text_from_folder(folder_path)

chunker = HybridSemanticChunker(embedding_model, breakpoint_ratio=0.15, merge_threshold=0.80)

docs = chunker.create_documents([full_document])

for d in docs:
    print(d.page_content)
    print("+++" * 30)


2026年4月20日

退職型株式給付信託（J-ESOP）導入による新株式発行、事後交付型株式報酬（RSU）制度に基づく新株式発行および過年度のRSU制度に基づく新株式発行の発行価額等の決定
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
事業概要

ネクセラファーマ株式会社（東証PRM 4565、以下、同社）は、英国での創薬プラットフォーム技術と、 日本およびアジア太平洋（中国を除く、以下、APAC）地域における開発・販売機能を融合させ、日本発・グローバル展開を目指すバイオ医薬品企業である。同社の売上収益は、世界の大手製薬会社との提携による収入と、自社品販売による収入の主に2つで構成される。
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
アッヴィ社との神経疾患を対象とした創薬提携において10百万米ドルのマイルストンを受領

退職型株式給付信託（J-ESOP）導入による新株式発行、事後交付型株式報酬（RSU）制度に基づく新株式発行および過年度のRSU制度に基づく新株式発行の発行価額等の決定
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
ネクセラファーマジャパン株式会社（出資比率100%）

Nxera Pharma Korea（旧・Idorsia Pharmaceuticals Korea Co., Ltd.、出資比率100%）
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
強み(Strengths)

複数の疾患に関連しているGPCRの構造解析を可能とするStaR技術の特許・ノウハウとNxWaveプラットフォームによる標的エンゲージメントの高い創薬力
+++++++++++++++++++++++++++++++++++++++++